# Normative GSE — Conto Termico: Dataset e Analisi

[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![Python 3.12](https://img.shields.io/badge/python-3.12-blue.svg)](https://www.python.org/downloads/)

**Autore:** Maurizio Lisanti — [PropLUG](https://proplug.it) — Linux User Group di Baronissi (SA)  
**Repository:** [github.com/MaurizioLisanti/conto-termico-gse](https://github.com/MaurizioLisanti/conto-termico-gse)  
**Dati:** Normative GSE estratte con Gemini API da PDF ufficiali (DM 16/02/2016, CT 3.0)

---

Questo notebook analizza il dataset strutturato delle normative del **Conto Termico GSE**,
il meccanismo italiano di incentivazione per l'efficienza energetica e le fonti rinnovabili.

L'intero processo — dall'estrazione PDF alla strutturazione JSON — è documentato qui in modo
riproducibile, **senza necessità di API key**: i dati sono già estratti in `data/`.


## Indice

1. [Introduzione al Conto Termico GSE](#1-introduzione)
2. [Processo di estrazione dai PDF](#2-estrazione)
3. [Esplorazione del dataset](#3-esplorazione)
4. [Esempi di utilizzo](#4-utilizzo)
5. [Conclusioni e risorse](#5-conclusioni)


---

## 1. Introduzione

### Cos'è il Conto Termico GSE

Il **Conto Termico** è il meccanismo di incentivazione italiano gestito dal GSE (Gestore dei Servizi Energetici)
che eroga contributi per:

- **Riqualificazione energetica** di edifici esistenti (isolamento, infissi, caldaie a condensazione, building automation)
- **Produzione di energia termica da fonti rinnovabili** (pompe di calore, solare termico, biomassa)

Il decreto di riferimento è il **DM 16/02/2016** (CT 2.0), ora in transizione verso il **CT 3.0** (2026).

### Perché questo dataset è utile

| Caso d'uso | Descrizione |
|---|---|
| **RAG su normativa** | Retrieval-Augmented Generation per assistenti AI che rispondono su CT GSE |
| **Fine-tuning LLM** | Coppie domanda-risposta su normativa italiana specialistica |
| **NLP italiano** | Testo normativo tecnico italiano per task di classificazione/NER |
| **Ricerca energetica** | Analisi strutturata delle politiche di incentivazione energetica |
| **Educazione** | Studio degli incentivi CT per tecnici e installatori |

### Fonti ufficiali utilizzate

Tutti i PDF sono documenti ufficiali GSE/MiTE disponibili pubblicamente:

| Documento | Tipo | Versione |
|---|---|---|
| DM 16/02/2016 | Decreto ministeriale | CT 2.0 |
| Allegato criteri ammissibilità | Allegato tecnico | CT 2.0 |
| Regole Applicative CT | Regole operative | CT 2.0 |
| Webinar CT 3.0 (2026) | Presentazione ufficiale | CT 3.0 |
| Guida PA | Guida operativa | CT 2.0 |
| Mappa interventi privati | Guida operativa | CT 2.0 |
| Regole pompe di calore | Regole tecniche | CT 2.0 |


---

## 2. Processo di estrazione

### Stack tecnologico

```
PDF ufficiali GSE
       │
       ▼
  pdfplumber          ← estrazione testo grezzi dai PDF
       │
       ▼
  Gemini API          ← strutturazione in JSON via prompt
(google-genai SDK)    
       │
       ▼
  dataset_completo.json ← output finale strutturato
```

**Dipendenze principali:**
- `pdfplumber` — estrazione testo da PDF
- `google-genai` — Gemini 2.0 Flash per strutturazione
- `json` — output strutturato

### Come funziona `extract_pdf.py`

Lo script principale segue questo flusso:

```python
# Pseudocodice semplificato di extract_pdf.py

def extract_pdf(pdf_path: str) -> dict:
    # 1. Estrai testo grezzo con pdfplumber
    with pdfplumber.open(pdf_path) as pdf:
        testo_grezzo = "\n".join(
            page.extract_text() for page in pdf.pages
        )
    
    # 2. Invia a Gemini per strutturazione
    prompt = PROMPT_TEMPLATE.format(testo=testo_grezzo)
    risposta = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt
    )
    
    # 3. Valida e restituisce il JSON strutturato
    return json.loads(risposta.text)
```

Il prompt istruisce Gemini a estrarre:
- **Sezioni** (titolo, contenuto, pagina, tag)
- **Interventi** (codice, descrizione, requisiti, incentivo, zone climatiche)
- **Tariffe** e **requisiti tecnici**
- **Citations** (riferimenti normativi)

### Nota: questo notebook non richiede API key

I dati sono già estratti e salvati in `data/dataset_completo.json`.
Tutte le celle seguenti utilizzano solo i dati pre-estratti.


In [ ]:
# Importazioni standard — nessuna API key richiesta
import json
import os
from pathlib import Path
from collections import Counter

# Gestione percorsi: locale vs Kaggle
KAGGLE_PATH = Path('/kaggle/input/conto-termico-gse-normative/dataset_completo.json')
LOCAL_PATH = Path('../data/dataset_completo.json')

if KAGGLE_PATH.exists():
    DATA_PATH = KAGGLE_PATH
else:
    DATA_PATH = LOCAL_PATH

print(f'Percorso dati: {DATA_PATH}')
print(f'File trovato: {DATA_PATH.exists()}')


In [ ]:
# Caricamento del dataset
with open(DATA_PATH, encoding='utf-8') as f:
    dataset = json.load(f)

documenti = dataset['documenti']
print(f'Dataset caricato: {len(documenti)} documenti')


---

## 3. Esplorazione del dataset

### 3.1 Struttura del dataset

Ogni documento nel dataset ha questa struttura:

```json
{
  "source": "nome_file.pdf",
  "tipo": "decreto | allegato | regole_applicative | guida",
  "data_documento": "YYYY-MM-DD",
  "versione": "CT 2.0 | CT 3.0",
  "sezioni": [
    {
      "titolo": "Titolo sezione",
      "contenuto": "Testo estratto...",
      "pagina": 3,
      "tags": ["keyword1", "keyword2"]
    }
  ],
  "interventi": [
    {
      "codice": "1.A",
      "descrizione": "Isolamento termico...",
      "requisiti": ["Trasmittanza massima..."],
      "incentivo": {},
      "zone_climatiche": ["A", "B", "C", "D", "E", "F"]
    }
  ],
  "tariffe": [],
  "requisiti_tecnici": ["..."],
  "citations": ["DM 16/02/2016", ...]
}
```


In [ ]:
# Statistiche generali del dataset
print('=' * 60)
print('STATISTICHE DATASET CONTO TERMICO GSE')
print('=' * 60)

totale_sezioni = sum(len(d.get('sezioni', [])) for d in documenti)
totale_interventi = sum(len(d.get('interventi', [])) for d in documenti)
totale_requisiti = sum(len(d.get('requisiti_tecnici', [])) for d in documenti)
totale_citations = sum(len(d.get('citations', [])) for d in documenti)

print(f'  Documenti totali     : {len(documenti)}')
print(f'  Sezioni totali       : {totale_sezioni}')
print(f'  Interventi totali    : {totale_interventi}')
print(f'  Requisiti tecnici    : {totale_requisiti}')
print(f'  Riferimenti normativi: {totale_citations}')
print()

# Distribuzione per tipo
tipo_counts = Counter(d['tipo'] for d in documenti)
print('Distribuzione per tipo documento:')
for tipo, count in sorted(tipo_counts.items()):
    print(f'  {tipo:25s} : {count}')
print()

# Distribuzione per versione CT
ver_counts = Counter(d['versione'] for d in documenti)
print('Distribuzione per versione CT:')
for ver, count in sorted(ver_counts.items()):
    print(f'  {ver:10s} : {count}')


In [ ]:
# Tabella riepilogativa dei documenti
print(f'{"Documento":<45} {"Tipo":<20} {"Ver":<7} {"Sez":>4} {"Int":>4}')
print('-' * 85)
for doc in documenti:
    nome = doc['source'].replace('.pdf', '')
    tipo = doc['tipo']
    ver = doc['versione']
    sez = len(doc.get('sezioni', []))
    intv = len(doc.get('interventi', []))
    print(f'{nome:<45} {tipo:<20} {ver:<7} {sez:>4} {intv:>4}')
print('-' * 85)
print(f'{"TOTALE":<45} {"":20} {"":7} {totale_sezioni:>4} {totale_interventi:>4}')


### 3.2 Interventi ammissibili

Il Conto Termico incentiva due categorie di interventi:

- **Titolo II / Art. 4 comma 1** — Riqualificazione energetica (solo PA)
- **Titolo III / Art. 4 comma 2** — Fonti rinnovabili e alta efficienza (PA e privati)


In [ ]:
# Raccoglie tutti gli interventi univoci dal dataset
all_interventi = []
for doc in documenti:
    for intv in doc.get('interventi', []):
        all_interventi.append({
            'codice': intv.get('codice', ''),
            'descrizione': intv.get('descrizione', ''),
            'n_requisiti': len(intv.get('requisiti', [])),
            'zone': ', '.join(intv.get('zone_climatiche', [])) or 'tutte',
            'fonte': doc['source'],
        })

# Mostra interventi con codici B.x (nomenclatura Regole Applicative)
print('INTERVENTI PRINCIPALI — REGOLE APPLICATIVE CT 2.0')
print('=' * 90)
print(f'{"Codice":<12} {"Descrizione":<60} {"Req":>4}')
print('-' * 90)

# Filtra interventi con codice numerico tipo '1.A', '2.A' ecc.
for item in all_interventi:
    codice = item['codice']
    # Mostra solo i codici delle Regole Applicative (1.x e 2.x)
    if codice and (codice.startswith('1.') or codice.startswith('2.')):
        desc = item['descrizione'][:57] + '...' if len(item['descrizione']) > 57 else item['descrizione']
        print(f'{codice:<12} {desc:<60} {item["n_requisiti"]:>4}')

print()
print(f'Totale record interventi nel dataset: {len(all_interventi)}')


In [ ]:
# Analisi sezioni: lunghezza testo per documento
print('ANALISI CONTENUTO SEZIONI')
print('=' * 70)

for doc in documenti:
    nome = doc['source'].replace('.pdf', '')
    sezioni = doc.get('sezioni', [])
    
    # Calcola testo totale estratto (escludi None)
    testo_totale = sum(
        len(s.get('contenuto') or '') 
        for s in sezioni
    )
    sezioni_piene = sum(
        1 for s in sezioni 
        if s.get('contenuto')
    )
    
    print(f'{nome}')
    print(f'  Sezioni totali: {len(sezioni):3d} | Con contenuto: {sezioni_piene:3d} | Caratteri: {testo_totale:,}')
    print()


In [ ]:
# Visualizzazione ASCII: distribuzione sezioni per documento
print('SEZIONI PER DOCUMENTO (grafico ASCII)')
print('-' * 60)

max_sezioni = max(len(d.get('sezioni', [])) for d in documenti)

for doc in documenti:
    nome = doc['source'].replace('_', ' ').replace('.pdf', '')
    n = len(doc.get('sezioni', []))
    barra = '█' * int(n * 30 / max_sezioni)
    print(f'{nome[:30]:<30} {barra:<30} {n:>3}')

print()
print('INTERVENTI PER DOCUMENTO')
print('-' * 60)

max_int = max(len(d.get('interventi', [])) for d in documenti)
for doc in documenti:
    nome = doc['source'].replace('_', ' ').replace('.pdf', '')
    n = len(doc.get('interventi', []))
    barra = '█' * int(n * 30 / max_int) if max_int > 0 else ''
    print(f'{nome[:30]:<30} {barra:<30} {n:>3}')


In [ ]:
# Analisi requisiti tecnici (documento più ricco)
print('REQUISITI TECNICI — allegato_criteri_ammissibilita')
print('=' * 60)

for doc in documenti:
    req = doc.get('requisiti_tecnici', [])
    if req:
        print(f'Fonte: {doc["source"]}')
        for i, r in enumerate(req, 1):
            print(f'  {i:2d}. {r}')
        print()

print('RIFERIMENTI NORMATIVI — allegato_criteri_ammissibilita')
print('=' * 60)

for doc in documenti:
    cit = doc.get('citations', [])
    if cit:
        print(f'Fonte: {doc["source"]}')
        for c in cit[:8]:  # mostra i primi 8
            print(f'  • {c}')
        if len(cit) > 8:
            print(f'  ... e altri {len(cit)-8} riferimenti')
        print()


---

## 4. Esempi di utilizzo

### 4.1 Query sul dataset

Esempi pratici per estrarre informazioni specifiche dal dataset JSON.


In [ ]:
# Query 1: Trova sezioni che parlano di pompe di calore
print('QUERY: sezioni contenenti "pompa di calore"')
print('=' * 60)

risultati = []
for doc in documenti:
    for sezione in doc.get('sezioni', []):
        contenuto = sezione.get('contenuto') or ''
        titolo = sezione.get('titolo', '')
        if 'pompa di calore' in contenuto.lower() or 'pompa di calore' in titolo.lower():
            risultati.append({
                'fonte': doc['source'],
                'titolo': titolo,
                'pagina': sezione.get('pagina', '?'),
                'preview': contenuto[:120] + '...' if len(contenuto) > 120 else contenuto,
            })

print(f'Trovati {len(risultati)} risultati:\n')
for r in risultati[:5]:  # mostra i primi 5
    print(f'  [{r["fonte"]} — p.{r["pagina"]}]')
    print(f'  {r["titolo"]}')
    if r['preview']:
        print(f'  {r["preview"]}')
    print()


In [ ]:
# Query 2: Tutti gli interventi con zone climatiche definite
print('QUERY: interventi con zone climatiche specificate')
print('=' * 60)

for doc in documenti:
    for intv in doc.get('interventi', []):
        zone = intv.get('zone_climatiche', [])
        if zone:
            codice = intv.get('codice', '')
            desc = intv.get('descrizione', '')[:50]
            print(f'  [{doc["source"][:30]}] {codice:15s} zone={zone} | {desc}')


In [ ]:
# Query 3: Estrai soggetti ammessi dal decreto
print('QUERY: articolo Soggetti Ammessi dal DM 16/02/2016')
print('=' * 60)

for doc in documenti:
    if 'decreto' in doc['tipo']:
        for sezione in doc.get('sezioni', []):
            if 'soggetti ammessi' in sezione.get('titolo', '').lower():
                print(f'Documento: {doc["source"]}')
                print(f'Titolo   : {sezione["titolo"]}')
                print(f'Pagina   : {sezione.get("pagina", "?")}\n')
                contenuto = sezione.get('contenuto', '') or ''
                print(contenuto[:400] + ('...' if len(contenuto) > 400 else ''))


### 4.2 Preparazione per RAG (Retrieval-Augmented Generation)

Questo dataset è ideale per sistemi RAG su normativa italiana.
Ecco come preparare i chunk per un vector store:


In [ ]:
# Preparazione chunk per RAG — nessuna API key necessaria
def crea_chunk_rag(dataset: dict) -> list[dict]:
    """
    Trasforma il dataset in chunk pronti per il vettorizzazione.
    Ogni chunk include metadata per il retrieval.
    """
    chunks = []
    
    for doc in dataset['documenti']:
        fonte = doc['source']
        versione = doc['versione']
        tipo = doc['tipo']
        
        # Chunk da sezioni
        for sezione in doc.get('sezioni', []):
            contenuto = sezione.get('contenuto') or ''
            if not contenuto.strip():
                continue  # salta sezioni senza testo
            
            chunk = {
                'id': f"{fonte}__p{sezione.get('pagina', 0)}",
                'text': contenuto,
                'metadata': {
                    'fonte': fonte,
                    'versione': versione,
                    'tipo': tipo,
                    'titolo_sezione': sezione.get('titolo', ''),
                    'pagina': sezione.get('pagina'),
                    'tags': sezione.get('tags', []),
                    'tipo_chunk': 'sezione',
                }
            }
            chunks.append(chunk)
        
        # Chunk da interventi (con requisiti)
        for intv in doc.get('interventi', []):
            req = intv.get('requisiti', [])
            if not req:
                continue
            
            testo_intv = f"Intervento {intv.get('codice', '')}: {intv.get('descrizione', '')}\n"
            testo_intv += 'Requisiti:\n' + '\n'.join(f'- {r}' for r in req)
            
            chunk = {
                'id': f"{fonte}__intv_{intv.get('codice', '').replace('.', '_')}",
                'text': testo_intv,
                'metadata': {
                    'fonte': fonte,
                    'versione': versione,
                    'tipo': tipo,
                    'codice_intervento': intv.get('codice', ''),
                    'zone_climatiche': intv.get('zone_climatiche', []),
                    'tipo_chunk': 'intervento',
                }
            }
            chunks.append(chunk)
    
    return chunks


# Genera i chunk
chunks = crea_chunk_rag(dataset)

print(f'Chunk RAG generati: {len(chunks)}')
print()

# Distribuzione per tipo chunk
tipo_chunk_counts = Counter(c['metadata']['tipo_chunk'] for c in chunks)
for tipo, count in tipo_chunk_counts.items():
    print(f'  {tipo:<15}: {count}')

print()
print('Esempio chunk (primo con contenuto):')
print('-' * 60)
example = chunks[0]
print(f'ID      : {example["id"]}')
print(f'Testo   : {example["text"][:200]}...')
print(f'Metadata: {json.dumps(example["metadata"], ensure_ascii=False, indent=2)}')


In [ ]:
# Statistiche chunk per RAG
print('STATISTICHE CHUNK RAG')
print('=' * 50)

lunghezze = [len(c['text']) for c in chunks]
if lunghezze:
    print(f'  Totale chunk       : {len(chunks)}')
    print(f'  Lunghezza media    : {sum(lunghezze)/len(lunghezze):.0f} caratteri')
    print(f'  Lunghezza min      : {min(lunghezze)} caratteri')
    print(f'  Lunghezza max      : {max(lunghezze)} caratteri')
    print()
    
    # Stima token (circa 4 char / token)
    token_medi = sum(lunghezze) / len(lunghezze) / 4
    print(f'  Token stimati/chunk: ~{token_medi:.0f}')
    print(f'  Token totali       : ~{sum(lunghezze)/4:.0f}')


### 4.3 Preparazione per fine-tuning LLM

Il dataset può essere usato per generare coppie `(domanda, risposta)` per il fine-tuning
di modelli LLM specializzati sulla normativa CT GSE.


In [ ]:
# Generazione esempi per fine-tuning da interventi con requisiti
def genera_esempi_finetuning(dataset: dict) -> list[dict]:
    """
    Crea coppie (istruzione, risposta) per fine-tuning
    dagli interventi con requisiti tecnici nel dataset.
    """
    esempi = []
    
    for doc in dataset['documenti']:
        for intv in doc.get('interventi', []):
            requisiti = intv.get('requisiti', [])
            if not requisiti:
                continue
            
            codice = intv.get('codice', '')
            descrizione = intv.get('descrizione', '')
            zone = intv.get('zone_climatiche', [])
            
            # Template domanda
            domanda = f"Quali sono i requisiti per l'intervento '{descrizione}' ({codice}) nel Conto Termico GSE?"
            
            # Risposta strutturata
            risposta_parts = [
                f"Per l'intervento {codice} — {descrizione}, i requisiti tecnici previsti dal Conto Termico GSE sono:",
                *[f"• {r}" for r in requisiti],
            ]
            if zone:
                risposta_parts.append(f"Zone climatiche applicabili: {', '.join(zone)}")
            risposta_parts.append(f"Fonte: {doc['source']} ({doc['versione']})")
            
            esempi.append({
                'instruction': domanda,
                'output': '\n'.join(risposta_parts),
                'source': doc['source'],
                'codice_intervento': codice,
            })
    
    return esempi


esempi_ft = genera_esempi_finetuning(dataset)

print(f'Esempi fine-tuning generati: {len(esempi_ft)}')
print()
print('Esempio 1:')
print('-' * 60)
if esempi_ft:
    ex = esempi_ft[0]
    print(f'ISTRUZIONE: {ex["instruction"]}')
    print()
    print(f'RISPOSTA:')
    print(ex['output'])


In [ ]:
# Secondo esempio
if len(esempi_ft) > 1:
    print('Esempio 2:')
    print('-' * 60)
    ex = esempi_ft[1]
    print(f'ISTRUZIONE: {ex["instruction"]}')
    print()
    print(f'RISPOSTA:')
    print(ex['output'])


### 4.4 Integrazione con Weaviate (RAG agentico)

Il progetto principale usa **Elysia** (framework RAG agentico di Weaviate)
per rispondere a domande sulla normativa CT GSE in tempo reale.

```python
# Esempio di integrazione con Weaviate (richiede .env con credenziali)
from tools import _verifica_ammissibilita, _stima_incentivo

# Verifica ammissibilità
risultato = _verifica_ammissibilita(
    tipo_impianto="pompa di calore aria-acqua",
    cop_certificato=3.2,
    zona_climatica="E"
)
# → {'ammissibile': True, 'tipo_intervento_ct': 'B.2 - Pompe di calore...', ...}

# Stima incentivo
stima = _stima_incentivo(
    tipo_impianto="pompa di calore",
    potenza_kw=12.0,
    tipo_soggetto="privato"
)
# → {'incentivo_annuo_eur': 1320.0, 'incentivo_totale_eur': 6600.0, 'durata_anni': 5, ...}
```

> **Nota:** Le funzioni sopra sono già implementate in `tools.py` con tariffe
> estratte direttamente da questo dataset. Vedi il repo per dettagli.


---

## 5. Conclusioni e risorse

### Cosa abbiamo visto

| Step | Descrizione | Output |
|---|---|---|
| Estrazione | pdfplumber → testo grezzo da 7 PDF GSE | ~72 sezioni |
| Strutturazione | Gemini 2.0 Flash → JSON strutturato | dataset_completo.json |
| Analisi | 7 documenti, 58 interventi, 20 requisiti tecnici | statistiche |
| RAG | Chunk da sezioni e interventi | chunk pronti per vettorizzazione |
| Fine-tuning | Coppie (domanda, risposta) da requisiti | esempi LLM |

### Repository GitHub

**[github.com/MaurizioLisanti/conto-termico-gse](https://github.com/MaurizioLisanti/conto-termico-gse)**

Il repo contiene:
- `extract_pdf.py` — script di estrazione con Gemini API
- `tools.py` — tool di ammissibilità e stima incentivo
- `main.py` — app RAG agentica con Elysia/Weaviate
- `tests/` — 36 test automatici (pytest)
- `data/` — dataset già estratto

### Come contribuire

Il progetto è sviluppato nell'ambito di **Linux PropLUG**,
l'associazione Linux di Baronissi (SA) che promuove il software libero nella comunità tecnica campana.

🌐 **[https://proplug.it](https://proplug.it)**

Contribuzioni benvenute:
1. Fai fork del repository
2. Aggiungi/migliora documenti GSE estratti
3. Apri una Pull Request

### Licenza

MIT License — vedi [LICENSE](../LICENSE) per i dettagli.

---

*Dataset creato da Maurizio Lisanti — PropLUG — Baronissi (SA), 2026.*  
*I PDF GSE sono documenti pubblici disponibili su [gse.it](https://www.gse.it).*


In [ ]:
# Riepilogo finale
print('=' * 60)
print('RIEPILOGO DATASET CONTO TERMICO GSE')
print('=' * 60)
print(f'  Documenti           : {len(documenti)}')
print(f'  Sezioni totali      : {totale_sezioni}')
print(f'  Interventi totali   : {totale_interventi}')
print(f'  Requisiti tecnici   : {totale_requisiti}')
print(f'  Riferimenti norm.   : {totale_citations}')
print(f'  Chunk RAG generati  : {len(chunks)}')
print(f'  Esempi fine-tuning  : {len(esempi_ft)}')
print()
print('Versioni coperte:')
for ver, count in sorted(ver_counts.items()):
    print(f'  {ver}: {count} documenti')
print()
print('Repository: https://github.com/MaurizioLisanti/conto-termico-gse')
print('Community : https://proplug.it')
print('Licenza   : MIT')
